# Network events: Request & Response

* When we navigate to a website, we send a request to a URL/website, and the website sends us a response back.
* The response can be an HTML code or other resources like CSS, JS, images, etc.
* To handle these request & response events, we set up event listeners provided by Playwright.

**Exampe**:

```python
from playwright.sync_api import Request, Response, Page, expect

def on_request(request: Request):
    print("Sent Request: ", request)

def on_response(response: Response):
    print("Got Response: ", response)


def test_doc_link(page: Page):
    page.on("request", on_request)
    page.on("response", on_response)

    page.goto("https://playwright.dev/python")

    docs_link = page.get_by_role("link", name="Docs")
    docs_link.click()

    expect(page).to_have_url("https://playwright.dev/python/docs/intro")
```

# Handle Requests

* Using an event listener, we cannot modify the request in any way before it's being sent.
* In order to modify or change the request, we need to use a **Route Handler**.

**Example**:

```python

from playwright.sync_api import Request, Response, Route, Page, expect

# Route object is page navigation object.
# It is the state of the navigation which has our request with some additional functionality to control our navigation.
# For example: continue, cancel, fulfill, etc.

def on_route(route: Route):
    # access the request object and can modify it
    request = route.request
    # request.post_data="mydata"

    # continue navigation
    # route.continue_()

    # cancel navigation
    print("Request aborted: ", request)
    route.abort()

# Abort loading all images
def on_img_check(route: Route):
    if route.request.resource_type == "image":
        print("Request aborted: ", route.request)
        route.abort()
    else:
        route.continue_()

def test_doc_link(page: Page):

    # Register Route Handler
    # 1st argument: URL to be handled (here we can use pattern matching like: **/*.svg)
    # 2nd argument: function to handle the route
    page.route(
        "https://playwright.dev/python/img/playwright-logo.svg",
        on_route
    )

    page.route(
        "**",   # select every URL
        on_img_check
    )

    page.goto("https://playwright.dev/python")
    page.screenshot(path="playwright.jpg")


```

# Returning custom Response

```python

from playwright.sync_api import Request, Response, Route, Page, expect

def on_route(route: Route):
    request = route.request
    print("Custom Response for Request: ", request)
    # return custom response
    route.fulfill(
        status=200,
        body="<html><body><h1>Custom Response!!!</h1></body></html>"
    )

def test_doc_link(page: Page):
    page.route(
        "https://playwright.dev/python",
        on_route
    )

    page.goto("https://playwright.dev/python")
    page.pause()
    page.screenshot(path="custom-response.jpg")


```

# Modify the actual response received

```python
from playwright.sync_api import Request, Response, Route, Page, expect

def modify_response(route: Route):
    response = route.fetch()
    body = response.text().replace(
        "Playwright enables reliable end-to-end testing for modern web apps.",
        "QA enables reliable end-to-end testing for modern web apps."
    )

    route.fulfill(
        response=response,
        body=body,
    )


def test_doc_link(page: Page):
    page.route(
        "https://playwright.dev/python",
        modify_response
    )

    page.goto("https://playwright.dev/python")
    page.screenshot(path="custom-response2.jpg")

```